# AML Graph Analysis & Results

This notebook computes:
1. Anomaly scores for all node embeddings
2. Graph statistics (degree distribution, connectivity)
3. Transaction value / money flow analysis
4. Suspicious node identification with financial impact

**Outputs saved to parquet** for downstream use by the chart exporter and dashboard.

In [ ]:
# Setup and imports
import os
import json
import numpy as np
import pandas as pd
import networkx as nx
from tensorflow import keras

# Define paths
BASE_PATH = os.path.dirname(os.path.abspath("__file__"))
TRAINING_DATA_PATH = os.path.join(BASE_PATH, "training_data")
OUTPUT_PATH = os.path.join(BASE_PATH, "output")
MODELS_PATH = os.path.join(BASE_PATH, "models")

# Override paths when running via pipeline (artifacts_dir injected by papermill)
try:
    if artifacts_dir:
        TRAINING_DATA_PATH = os.path.join(artifacts_dir, "data")
        OUTPUT_PATH = os.path.join(artifacts_dir, "data")
        MODELS_PATH = os.path.join(artifacts_dir, "models")
except NameError:
    pass

os.makedirs(OUTPUT_PATH, exist_ok=True)

print("Libraries loaded successfully!")

## 1. Load Data

In [ ]:
# Load edges (transactions)
edges_df = pd.read_csv(os.path.join(TRAINING_DATA_PATH, "edges_td.csv"))
print(f"Loaded {len(edges_df)} edges (transactions)")
print(f"Edge columns: {edges_df.columns.tolist()}")
edges_df.head()

Loaded 438386 edges (transactions)
Edge columns: ['source', 'target', 'tran_id', 'tx_type', 'base_amt']


,source,target,tran_id,tx_type,base_amt
0,3aa9646b,1e46e726,496,4,858.77
1,49203bc3,a74d1101,1342,4,386.86
2,616d4505,99af2455,1580,4,616.43
3,39be1ea2,e7ec7bdb,2866,4,146.44
4,e2e0d938,afc399a9,3997,4,439.09


In [ ]:
# Load nodes (parties)
nodes_df = pd.read_csv(os.path.join(TRAINING_DATA_PATH, "node_td.csv"))
print(f"Loaded {len(nodes_df)} nodes (parties)")
print(f"Node columns: {nodes_df.columns.tolist()}")
nodes_df.head()

Loaded 7347 nodes (parties)
Node columns: ['id', 'type']


,id,type
0,5628bd6c,0
1,a1fcba39,0
2,f56c9501,1
3,9969afdd,0
4,b356eeae,1


In [ ]:
# Load node embeddings with anomaly scores
node_embeddings = pd.read_parquet(os.path.join(OUTPUT_PATH, "node_embeddings_fg.parquet"))
print(f"Loaded {len(node_embeddings)} node embeddings")

# Load the trained model to compute anomaly scores
model_dirs = [d for d in os.listdir(MODELS_PATH) if d.startswith('gan_anomaly_')]
latest_model_dir = os.path.join(MODELS_PATH, sorted(model_dirs)[-1])
model = keras.models.load_model(os.path.join(latest_model_dir, "anomaly_detector.keras"))
threshold = np.load(os.path.join(latest_model_dir, "threshold.npy"))

print(f"Model loaded from: {latest_model_dir}")
print(f"Anomaly threshold: {threshold:.6f}")

Loaded 7347 node embeddings
Model loaded from: /home/adnoman/projects/aml_gan/AMLend2end/models/gan_anomaly_5f5ae592
Anomaly threshold: 0.000152


In [ ]:
# Compute anomaly scores for all nodes
emb_cols = [c for c in node_embeddings.columns if c.startswith('emb_')]
all_embeddings = node_embeddings[emb_cols].values
reconstructed = model.predict(all_embeddings, verbose=0)
anomaly_scores = np.mean(np.square(all_embeddings - reconstructed), axis=1)

# Add to dataframe
node_embeddings['anomaly_score'] = anomaly_scores
node_embeddings['is_anomaly'] = anomaly_scores > threshold

# Normalize to 0-1 risk score
score_min, score_max = anomaly_scores.min(), anomaly_scores.max()
if score_max > score_min:
    node_embeddings['risk_score'] = (anomaly_scores - score_min) / (score_max - score_min)
else:
    node_embeddings['risk_score'] = 0.0

# Save enriched embeddings back to parquet (with anomaly_score, is_anomaly, risk_score)
out_path = os.path.join(OUTPUT_PATH, 'node_embeddings_fg.parquet')
node_embeddings.to_parquet(out_path, index=False)

print(f"Anomaly scores computed!")
print(f"Score range: [{anomaly_scores.min():.6f}, {anomaly_scores.max():.6f}]")
print(f"Threshold: {threshold:.6f}")
print(f"Anomalies: {node_embeddings['is_anomaly'].sum()} / {len(node_embeddings)}")
print(f"Saved enriched embeddings to {out_path}")

In [ ]:
# Create NetworkX graph
G = nx.from_pandas_edgelist(
    edges_df,
    source='source',
    target='target',
    create_using=nx.DiGraph()
)

print(f"Graph Statistics:")
print(f"  Nodes: {G.number_of_nodes()}")
print(f"  Edges: {G.number_of_edges()}")
print(f"  Density: {nx.density(G):.4f}")
print(f"  Is connected: {nx.is_weakly_connected(G)}")
print(f"  Number of connected components: {nx.number_weakly_connected_components(G)}")

Graph Statistics:
  Nodes: 7347
  Edges: 17070
  Density: 0.0003
  Is connected: False
  Number of connected components: 5


## 2. Build Network Graph

## 2.5 Transaction Value Analysis

In [ ]:
# Transaction amount analysis
print("Transaction Amount Statistics:")
print("=" * 50)
print(f"Total transactions: {len(edges_df):,}")
print(f"Total transaction value: ${edges_df['base_amt'].sum():,.2f}")
print(f"Average transaction: ${edges_df['base_amt'].mean():,.2f}")
print(f"Median transaction: ${edges_df['base_amt'].median():,.2f}")
print(f"Min transaction: ${edges_df['base_amt'].min():,.2f}")
print(f"Max transaction: ${edges_df['base_amt'].max():,.2f}")
print("=" * 50)

In [ ]:
# Calculate money flow per node
outgoing_amounts = edges_df.groupby('source')['base_amt'].sum().rename('outgoing_amt')
incoming_amounts = edges_df.groupby('target')['base_amt'].sum().rename('incoming_amt')
outgoing_counts = edges_df.groupby('source').size().rename('outgoing_count')
incoming_counts = edges_df.groupby('target').size().rename('incoming_count')

# Merge with node embeddings
node_money = node_embeddings[['id', 'anomaly_score', 'is_anomaly']].copy()
if 'is_sar' in node_embeddings.columns:
    node_money['is_sar'] = node_embeddings['is_sar']

node_money = node_money.merge(outgoing_amounts, left_on='id', right_index=True, how='left')
node_money = node_money.merge(incoming_amounts, left_on='id', right_index=True, how='left')
node_money = node_money.merge(outgoing_counts, left_on='id', right_index=True, how='left')
node_money = node_money.merge(incoming_counts, left_on='id', right_index=True, how='left')
node_money = node_money.fillna(0)

# Calculate total volume and net flow
node_money['total_volume'] = node_money['outgoing_amt'] + node_money['incoming_amt']
node_money['net_flow'] = node_money['incoming_amt'] - node_money['outgoing_amt']
node_money['total_transactions'] = node_money['outgoing_count'] + node_money['incoming_count']

print("Money Flow Statistics:")
print("=" * 50)
print(f"Total money volume: ${node_money['total_volume'].sum()/2:,.2f}")  # Divide by 2 to avoid double counting
print(f"Avg volume per node: ${node_money['total_volume'].mean():,.2f}")
print(f"Max volume (single node): ${node_money['total_volume'].max():,.2f}")
print("=" * 50)

Money Flow Statistics:
Total money volume: $242,434,979.01
Avg volume per node: $65,995.64
Max volume (single node): $452,657.49


In [ ]:
# Compare money flow: Anomalous vs Normal nodes
anomalous_money = node_money[node_money['is_anomaly'] == True]
normal_money = node_money[node_money['is_anomaly'] == False]

print("Money Flow Comparison: Anomalous vs Normal Nodes")
print("=" * 60)
print(f"{'Metric':<30} {'Anomalous':>15} {'Normal':>15}")
print("-" * 60)
print(f"{'Node Count':<30} {len(anomalous_money):>15,} {len(normal_money):>15,}")
print(f"{'Total Volume ($)':<30} {anomalous_money['total_volume'].sum():>15,.0f} {normal_money['total_volume'].sum():>15,.0f}")
print(f"{'Avg Volume per Node ($)':<30} {anomalous_money['total_volume'].mean():>15,.0f} {normal_money['total_volume'].mean():>15,.0f}")
print(f"{'Avg Transactions per Node':<30} {anomalous_money['total_transactions'].mean():>15,.1f} {normal_money['total_transactions'].mean():>15,.1f}")
print(f"{'Avg Outgoing ($)':<30} {anomalous_money['outgoing_amt'].mean():>15,.0f} {normal_money['outgoing_amt'].mean():>15,.0f}")
print(f"{'Avg Incoming ($)':<30} {anomalous_money['incoming_amt'].mean():>15,.0f} {normal_money['incoming_amt'].mean():>15,.0f}")
print("=" * 60)

In [ ]:
# Top 20 suspicious nodes by transaction volume
top_suspicious_volume = node_money[node_money['is_anomaly'] == True].nlargest(20, 'total_volume')

print("Top 20 Suspicious Nodes by Transaction Volume:")
print("=" * 80)
cols_to_show = ['id', 'total_volume', 'outgoing_amt', 'incoming_amt', 'total_transactions', 'anomaly_score']
if 'is_sar' in top_suspicious_volume.columns:
    cols_to_show.insert(1, 'is_sar')
print(top_suspicious_volume[cols_to_show].to_string(index=False))

# Total suspicious money
total_suspicious = anomalous_money['total_volume'].sum() / 2  # Divide by 2 to avoid double counting
print(f"\nTotal money flowing through suspicious nodes: ${total_suspicious:,.2f}")

In [ ]:
# Money flow network summary (visualization handled by chart exporter)
top_nodes = node_money.nlargest(50, 'total_volume')['id'].tolist()
suspicious_high_volume = node_money[(node_money['is_anomaly']) & (node_money['total_volume'] > node_money['total_volume'].median())]['id'].tolist()[:50]
vis_nodes = set(top_nodes + suspicious_high_volume)

# Filter to nodes present in graph
vis_in_graph = [n for n in vis_nodes if n in G.nodes()]
G_money_vis = G.subgraph(vis_in_graph)

print(f"Money Flow Network Summary:")
print(f"  High-volume + suspicious nodes: {len(vis_nodes)}")
print(f"  In graph: {G_money_vis.number_of_nodes()} nodes, {G_money_vis.number_of_edges()} edges")

In [ ]:
# Calculate node degrees
in_degrees = dict(G.in_degree())
out_degrees = dict(G.out_degree())
total_degrees = {n: in_degrees.get(n, 0) + out_degrees.get(n, 0) for n in G.nodes()}

print(f"Degree Statistics:")
print(f"  In-degree  — mean: {np.mean(list(in_degrees.values())):.2f}, max: {max(in_degrees.values())}")
print(f"  Out-degree — mean: {np.mean(list(out_degrees.values())):.2f}, max: {max(out_degrees.values())}")
print(f"  Total      — mean: {np.mean(list(total_degrees.values())):.2f}, max: {max(total_degrees.values())}")

print(f"\nTop 10 nodes by total degree:")
top_degree = sorted(total_degrees.items(), key=lambda x: x[1], reverse=True)[:10]
for node, degree in top_degree:
    print(f"  {node}: {degree}")

## 3. Anomaly Score Analysis

In [ ]:
# Anomaly score distribution summary
print("Anomaly Score Distribution:")
print(f"  Mean:   {node_embeddings['anomaly_score'].mean():.6f}")
print(f"  Median: {node_embeddings['anomaly_score'].median():.6f}")
print(f"  Std:    {node_embeddings['anomaly_score'].std():.6f}")
print(f"  Min:    {node_embeddings['anomaly_score'].min():.6f}")
print(f"  Max:    {node_embeddings['anomaly_score'].max():.6f}")
print(f"  Threshold: {threshold:.6f}")

if 'is_sar' in node_embeddings.columns:
    sar_scores = node_embeddings[node_embeddings['is_sar'] == 1]['anomaly_score']
    non_sar_scores = node_embeddings[node_embeddings['is_sar'] == 0]['anomaly_score']
    print(f"\n  SAR nodes — mean score: {sar_scores.mean():.6f}, median: {sar_scores.median():.6f}")
    print(f"  Non-SAR   — mean score: {non_sar_scores.mean():.6f}, median: {non_sar_scores.median():.6f}")

In [ ]:
# Summary statistics
total_nodes = len(node_embeddings)
anomaly_count = node_embeddings['is_anomaly'].sum()
normal_count = total_nodes - anomaly_count

print("=" * 50)
print("ANOMALY DETECTION SUMMARY")
print("=" * 50)
print(f"Total nodes analyzed: {total_nodes}")
print(f"Anomalies detected:   {anomaly_count} ({100*anomaly_count/total_nodes:.1f}%)")
print(f"Normal nodes:         {normal_count} ({100*normal_count/total_nodes:.1f}%)")
print(f"\nAnomaly threshold:    {threshold:.6f}")
print(f"Mean anomaly score:   {node_embeddings['anomaly_score'].mean():.6f}")
print(f"Median anomaly score: {node_embeddings['anomaly_score'].median():.6f}")
print(f"Std anomaly score:    {node_embeddings['anomaly_score'].std():.6f}")
print("=" * 50)

# If SAR labels exist, show confusion matrix-like stats
if 'is_sar' in node_embeddings.columns:
    sar_nodes = node_embeddings['is_sar'].sum()
    detected_sar = ((node_embeddings['is_sar'] == 1) & (node_embeddings['is_anomaly'])).sum()
    print(f"\nSAR Labels in data:   {sar_nodes}")
    print(f"SAR detected as anomaly: {detected_sar} ({100*detected_sar/max(sar_nodes,1):.1f}%)")

ANOMALY DETECTION SUMMARY
Total nodes analyzed: 7347
Anomalies detected:   7303 (99.4%)
Normal nodes:         44 (0.6%)

Anomaly threshold:    0.000152
Mean anomaly score:   0.000276
Median anomaly score: 0.000275
Std anomaly score:    0.000052

SAR Labels in data:   816
SAR detected as anomaly: 811 (99.4%)


## 4. Visualize Transaction Network Graph

In [ ]:
# Network graph sampling summary (visualization handled by chart exporter)
MAX_NODES_VIS = 500

node_scores = node_embeddings.set_index('id')['anomaly_score'].to_dict()
anomalous_nodes = node_embeddings[node_embeddings['is_anomaly']]['id'].tolist()
anomalous_in_graph = [n for n in anomalous_nodes if n in G.nodes()]

print(f"Network Graph Summary:")
print(f"  Total graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
print(f"  Anomalous nodes in graph: {len(anomalous_in_graph)}")
print(f"  High-degree nodes (degree > 10): {sum(1 for d in total_degrees.values() if d > 10)}")

In [ ]:
# Node attribute summary
node_anomaly = node_embeddings.set_index('id')['is_anomaly'].to_dict()
anomaly_count_in_graph = sum(1 for n in G.nodes() if node_anomaly.get(n, False))
print(f"Nodes with anomaly flag in graph: {anomaly_count_in_graph} / {G.number_of_nodes()}")

In [ ]:
# Network graph stats (visualization handled by chart exporter)
print(f"Graph visualization data prepared")
print(f"  Anomalous nodes: {anomaly_count_in_graph}")
print(f"  Normal nodes: {G.number_of_nodes() - anomaly_count_in_graph}")

## 5. Top Anomalous Nodes Analysis

In [ ]:
# Top 20 most anomalous nodes
top_anomalies = node_embeddings.nlargest(20, 'anomaly_score')[['id', 'anomaly_score', 'is_anomaly']]
if 'is_sar' in node_embeddings.columns:
    top_anomalies = node_embeddings.nlargest(20, 'anomaly_score')[['id', 'is_sar', 'anomaly_score', 'is_anomaly']]

print("Top 20 Most Anomalous Nodes:")
print("=" * 60)
print(top_anomalies.to_string(index=False))

In [ ]:
# Top anomalous node neighborhood summary
top_node = top_anomalies.iloc[0]['id']

if top_node in G.nodes():
    # Get 2-hop neighborhood
    neighbors_1 = set(G.predecessors(top_node)) | set(G.successors(top_node))
    neighbors_2 = set()
    for n in neighbors_1:
        neighbors_2 |= set(G.predecessors(n)) | set(G.successors(n))
    
    subgraph_nodes = {top_node} | neighbors_1 | neighbors_2
    G_sub = G.subgraph(subgraph_nodes).copy()
    
    print(f"Top anomaly '{top_node}' neighborhood:")
    print(f"  1-hop neighbors: {len(neighbors_1)}")
    print(f"  2-hop neighbors: {len(neighbors_2 - neighbors_1 - {top_node})}")
    print(f"  Subgraph: {G_sub.number_of_nodes()} nodes, {G_sub.number_of_edges()} edges")
else:
    print(f"Node {top_node} not found in graph")

## 6. Summary Report

In [ ]:
# Final summary
print("\n" + "=" * 70)
print(" AML ANOMALY DETECTION - ANALYSIS REPORT ")
print("=" * 70)

print(f"\n{'Graph Statistics':^40}")
print("-" * 40)
print(f"  Total Nodes:          {G.number_of_nodes():,}")
print(f"  Total Edges:          {G.number_of_edges():,}")
print(f"  Graph Density:        {nx.density(G):.6f}")
print(f"  Avg Degree:           {sum(total_degrees.values())/len(total_degrees):.2f}")

print(f"\n{'Transaction Value Statistics':^40}")
print("-" * 40)
print(f"  Total Transactions:   {len(edges_df):,}")
print(f"  Total Value:          ${edges_df['base_amt'].sum():,.2f}")
print(f"  Average Transaction:  ${edges_df['base_amt'].mean():,.2f}")
print(f"  Max Transaction:      ${edges_df['base_amt'].max():,.2f}")

print(f"\n{'Anomaly Detection Results':^40}")
print("-" * 40)
print(f"  Nodes Analyzed:       {len(node_embeddings):,}")
print(f"  Anomalies Detected:   {node_embeddings['is_anomaly'].sum():,} ({100*node_embeddings['is_anomaly'].mean():.1f}%)")
print(f"  Detection Threshold:  {threshold:.6f}")
print(f"  Max Anomaly Score:    {node_embeddings['anomaly_score'].max():.6f}")

print(f"\n{'Suspicious Money Flow':^40}")
print("-" * 40)
suspicious_volume = node_money[node_money['is_anomaly']]['total_volume'].sum() / 2
total_volume = edges_df['base_amt'].sum()
print(f"  Suspicious Volume:    ${suspicious_volume:,.2f}")
print(f"  % of Total Volume:    {100*suspicious_volume/total_volume:.1f}%")
print(f"  Avg per Suspicious:   ${node_money[node_money['is_anomaly']]['total_volume'].mean():,.2f}")

print(f"\n{'Data Outputs Saved':^40}")
print("-" * 40)
print(f"  - node_embeddings_fg.parquet (with anomaly_score, is_anomaly, risk_score)")

print("\n" + "=" * 70)
print(" ANALYSIS COMPLETE ")
print("=" * 70)